In [4]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

In [ ]:
from qdrant_client.models import VectorParams, Distance

if not client.collection_exists("news_paper"):
   client.create_collection(
      collection_name="news_paper",
      vectors_config=VectorParams(size=512, distance=Distance.COSINE),
   )

In [6]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\Pavan VK Alapati\Downloads\search_engine\data\Free-Press-Mumbai-Epaper-18-11-2025.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                                   chunk_overlap=50)
texts = text_splitter.split_documents(documents)

# Load the embedding model 
model_name = "BAAI/bge-large-en"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [14]:
print(texts[0].metadata)

{'producer': 'itext-paulo-155 (itextpdf.sf.net-lowagie.com)', 'creator': 'pdftk-java 3.2.2', 'creationdate': '2025-11-17T23:40:04+00:00', 'moddate': '2025-11-17T23:40:04+00:00', 'source': 'C:\\Users\\Pavan VK Alapati\\Downloads\\search_engine\\data\\Free-Press-Mumbai-Epaper-18-11-2025.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}


In [17]:
docs = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]
ids = [i for i in range(len(texts))]

In [19]:
client.add(collection_name="search",documents=docs,metadata=metadata,ids=ids)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\anaconda3\envs\search_engine\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pavan VK Alapati\AppData\Local\Temp\fastembed_cache\models--Qdrant--bge-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 5 files: 100%|██████████| 5/5 

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [26]:
search_result = client.query(collection_name="search",query_text="CNG COLLAPSE CRIPPLES CITY",limit=10)
for result in search_result:
    
   print(result.metadata['document'])

ANAL YSIS
Will have little effect
Scan & Watch
CNG COLLAPSE CRIPPLES CITY
Kamal Mishra  
MUMBAI 
Mumbai was thrown out of 
gear on Monday after a major 
disruption in CNG supply 
brought thousands of autos, 
taxis, school buses, and other 
gas-run vehicles to a halt.  
An MGL spokesperson said 
the supply to their City Gate 
Station (CGS) at Wadala was 
disrupted on Sunday due to 
damage to a GAIL main gas 
pipeline within the Rashtriya 
Chemicals and Fertisiers Ltd’s
ruption, owing to damage to a
key GAIL pipeline in RCF, was
witnessed on Monday across
the Mumbai Metropolitan
Region (MMR). With many
autos off the roads, com-
muters were squeezed into
overcrowded buses and local
trains. Several people
reported delays of two to
three hours on regular office
routes. 
The president of School
Bus Owners Association, Anil
Garg said operators faced
major problems due to the
non-availability of CNG.
Nearly 2,000 school buses
were stranded on Monday,
the industrial areas and on 
the Bidkin–Chi

## initiating client

In [2]:
pip install zep-cloud openai rich python-dotenv

  Using cached openai-2.8.1-py3-none-any.whl.metadata (29 kB)
  Using cached rich-14.2.0-py3-none-any.whl.metadata (18 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached openai-2.8.1-py3-none-any.whl (1.0 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.12.0-cp312-cp312-win_amd64.whl (205 kB)
Using cached rich-14.2.0-py3-none-any.whl (243 kB)
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)

   ----- ---------------------------------- 1/7 [jiter]
   ----------- ---------------------------- 2/7 [distro]
   ----------------- ---------------------- 3/7 [markdown-it-py]
   ----------------- ---------------------- 3/7 [markdown-it-py]
   ----------------- ------------------

In [3]:
import os
import json
import uuid

from openai import OpenAI
import rich

from dotenv import load_dotenv
from zep_cloud.client import Zep
from zep_cloud import Message

load_dotenv()

zep = Zep(api_key=os.environ.get("ZEP_API_KEY"))

oai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)


### creating a thread

In [4]:
bot_name = "zeptester"
user_name = "abc"
user_id = user_name + str(uuid.uuid4())[:4]
thread_id = str(uuid.uuid4())

zep.user.add(
    user_id=user_id,
    email=f"{user_name}@painters.com",
    first_name=user_name,
    last_name="Painter",
)

zep.thread.create(
    user_id=user_id,
    thread_id=thread_id,
)


Thread(created_at='2025-11-20T15:53:49.309724Z', project_uuid='00000000-0000-0000-0000-000000000000', thread_id='967650f0-6e46-4a86-bd81-b4b53dcbbd48', user_id='abce90d', uuid_='d366d3ea-5ffe-4cd2-9546-81e55dc04071')

### Conversation Data

In [5]:
support_cases = [
    {
        "subject": "Bug: Magic Pen Tool Drawing Goats Instead of Boats",
        "messages": [
            {
                "role": "user",
                "content": "Whenever I use the magic pen tool to draw boats, it ends up drawing goats instead.",
                "timestamp": "2024-03-16T14:20:00Z",
            },
            {
                "role": "support_agent",
                "content": f"Hi {user_name}, that sounds like a bug! Thanks for reporting it. Could you let me know exactly how you're using the tool when this happens?",
                "timestamp": "2024-03-16T14:22:00Z",
            },
            {
                "role": "user",
                "content": "Sure, I select the magic pen, draw a boat shape, and it just replaces the shape with goats.",
                "timestamp": "2024-03-16T14:25:00Z",
            },
            {
                "role": "support_agent",
                "content": "Got it! We'll escalate this to our engineering team. In the meantime, you can manually select the boat shape from the options rather than drawing it with the pen.",
                "timestamp": "2024-03-16T14:27:00Z",
            },
            {
                "role": "user",
                "content": "Okay, thanks. I hope it gets fixed soon!",
                "timestamp": "2024-03-16T14:30:00Z",
            },
        ],
        "status": "escalated",
    },
]

chat_history = [
    {
        "role": "assistant",
        "name": bot_name,
        "content": f"Hello {user_name}, welcome to PaintWiz support. How can I assist you today?",
        "timestamp": "2024-03-15T10:00:00Z",
    },
    {
        "role": "user",
        "name": user_name,
        "content": "I'm absolutely furious! Your AI art generation is completely broken!",
        "timestamp": "2024-03-15T10:02:00Z",
    },
    {
        "role": "assistant",
        "name": bot_name,
        "content": f"I'm sorry to hear that you're experiencing issues, {user_name}. Can you please provide more details about what's going wrong?",
        "timestamp": "2024-03-15T10:03:00Z",
    },
    {
        "role": "user",
        "name": user_name,
        "content": "Every time I try to draw mountains, your stupid app keeps turning them into fountains! And what's worse, all the people in my drawings have six fingers! It's ridiculous!",
        "timestamp": "2024-03-15T10:05:00Z",
    },
    {
        "role": "assistant",
        "name": bot_name,
        "content": f"I sincerely apologize for the frustration this is causing you, {user_name}. That certainly sounds like a significant glitch in our system. I understand how disruptive this can be to your artistic process. Can you tell me which specific tool or feature you're using when this occurs?",
        "timestamp": "2024-03-15T10:06:00Z",
    },
    {
        "role": "user",
        "name": user_name,
        "content": "I'm using the landscape generator and the character creator. Both are completely messed up. How could you let this happen?",
        "timestamp": "2024-03-15T10:08:00Z",
    },
]

transactions = [
    {
        "date": "2024-07-30",
        "amount": 99.99,
        "status": "Success",
        "account_id": user_id,
        "card_last_four": "1234",
    },
    {
        "date": "2024-08-30",
        "amount": 99.99,
        "status": "Failed",
        "account_id": user_id,
        "card_last_four": "1234",
        "failure_reason": "Card expired",
    },
    {
        "date": "2024-09-15",
        "amount": 99.99,
        "status": "Failed",
        "account_id": user_id,
        "card_last_four": "1234",
        "failure_reason": "Card expired",
    },
]

account_status = {
    "user_id": user_id,
    "account": {
        "account_id": user_id,
        "account_status": {
            "status": "suspended",
            "reason": "payment failure",
        },
    },
}

def convert_to_zep_messages(chat_history: list[dict[str, str | None]]) -> list[Message]:
    """
    Convert chat history to Zep messages.

    Args:
    chat_history (list): List of dictionaries containing chat messages.

    Returns:
    list: List of Zep Message objects.
    """
    return [
        Message(
            role=msg["role"],
            name=msg.get("name", None),
            content=msg["content"],
        )
        for msg in chat_history
    ]

# Zep's high-level API allows us to add a list of messages to a thread.
zep.thread.add_messages(
    thread_id=thread_id, messages=convert_to_zep_messages(chat_history)
)

# The lower-level data API allows us to add arbitrary data to a user's Knowledge Graph.
for tx in transactions:
    zep.graph.add(user_id=user_id, data=json.dumps(tx), type="json")

    zep.graph.add(
        user_id=user_id, data=json.dumps(account_status), type="json"
    )

for case in support_cases:
    zep.graph.add(user_id=user_id, data=json.dumps(case), type="json")


### Data Retrival from ZEP

In [11]:
all_user_edges = zep.graph.edge.get_by_user_id(user_id=user_id)
rich.print(all_user_edges[:])

[
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:57:15.897245Z',
        episodes=['562a9c5f-0b00-47c8-b42b-2a09b792891d'],
        expired_at=None,
        fact='The support agent suggested the user manually select the boat shape instead of using the pen.',
        invalid_at=None,
        name='SUGGESTS_WORKAROUND',
        relevance=None,
        score=None,
        source_node_uuid='80ae1757-97ee-43c1-898b-ba176e3ef2ac',
        target_node_uuid='0db9b87a-a1ae-4a7a-8bdd-025a40d7a197',
        uuid_='faeaebfb-bfc1-4b43-85ae-6f0887e23ac6',
        valid_at='2024-03-16T14:27:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:57:15.897214Z',
        episodes=['562a9c5f-0b00-47c8-b42b-2a09b792891d'],
        expired_at=None,
        fact='The user specifies that they select the magic pen and draw a boat shape.',
        invalid_at=None,
        name='CLARIFIES_ACTION',
        relevance=None,
        score=None,
        source_node_uuid='e1076023-b90b-4200-9eca-37b8b353386e',
        target_node_uuid='befe21c2-2813-4e03-b731-e82d74680b14',
        uuid_='f885fff4-cde2-4ec2-86c1-427c9cc8525b',
        valid_at='2024-03-16T14:25:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:56:31.690037Z',
        episodes=['1127acb2-ca64-4e13-8921-7008ab279b04'],
        expired_at=None,
        fact='The amount 99.99 was associated with the failure reason Card expired.',
        invalid_at=None,
        name='FAILURE_REASON_FOR_AMOUNT',
        relevance=None,
        score=None,
        source_node_uuid='3267f2e3-c2c1-4d02-89a4-c4adf2134c8f',
        target_node_uuid='58793d96-f94e-4cb6-b1fa-f5b04a93f3d7',
        uuid_='f5b72ffa-092b-4ad4-88ab-4e5af1998b40',
        valid_at='2024-08-30T00:00:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:56:13.497Z',
        episodes=['d473e456-ac49-4fd3-ab97-70f014fd4733'],
        expired_at='2025-11-20T15:56:34.166992Z',
        fact='The transaction involving account abce90d on 2024-07-30 had a status of Success.',
        invalid_at='2024-08-30T00:00:00Z',
        name='HAS_STATUS',
        relevance=None,
        score=None,
        source_node_uuid='e1076023-b90b-4200-9eca-37b8b353386e',
        target_node_uuid='d22f2a0d-59a0-4055-a2df-fac3460eb0c1',
        uuid_='f0aed6f3-90df-4917-9d74-f8121974e59d',
        valid_at='2024-07-30T00:00:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:56:22.843Z',
        episodes=[
            'ade3e003-6f92-47ff-a15e-2df0c313ce00',
            '5e03d946-3479-428f-906b-31539e2da43a',
            '16310199-5748-4821-814d-8ba7df4ccc53'
        ],
        expired_at=None,
        fact='The reason for the account suspension is a payment failure.',
        invalid_at=None,
        name='HAS_STATUS_REASON',
        relevance=None,
        score=None,
        source_node_uuid='22f81d58-504d-44c5-bb1d-ba2f8f110d5d',
        target_node_uuid='ee45e156-0fcb-4d14-8d0b-6df071e59b4c',
        uuid_='eb8e1428-c5dd-4cd4-92d0-c45a50b078e3',
        valid_at='2025-11-20T15:54:57.999Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:56:51.299505Z',
        episodes=['254bb73e-c974-4b07-8749-c40efd163425'],
        expired_at=None,
        fact="The 'Failed' status is linked to account abce90d.",
        invalid_at=None,
        name='RELATES_TO_ACCOUNT',
        relevance=None,
        score=None,
        source_node_uuid='59812e79-26c6-4faa-af08-31bc0a8c0f68',
        target_node_uuid='13f9d406-2a5f-4af4-9183-c3fe4fac4785',
        uuid_='e4386521-bc9b-4f1d-9ed1-83f35438234a',
        valid_at='2024-0

In [7]:
memory = zep.thread.get_user_context(thread_id=thread_id)
rich.print(memory.context)

# This is the user summary
<USER_SUMMARY>
The user's name is abc Painter, and their email address is abc@painters.com. Their user ID is abce90d.

The user highly values functional AI art generation, having expressed extreme frustration when this feature was 
broken.

The user has not provided any specific procedural instructions for the AI's assistance.
</USER_SUMMARY>


FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
# NOTE: Facts ending in "present" are currently valid (e.g., "Jane prefers her coffee with milk (2024-01-15 
10:30:00 - present)" means Jane currently prefers coffee with milk)
#       Facts with a past end date used to be valid but are NOT CURRENTLY VALID (e.g., "Jane prefers her coffee 
with milk (2024-01-15 10:30:00 - 2024-06-20 14:00:00)" means Jane no longer prefers coffee with milk)
<FACTS>
  - The glitch in the system is causing disruption to abc Painter's artistic process. (2025-11-20 15:54:57 - 
present)
  - zeptester asked abc Painter to specify which tool or feature is causing the issue. (2025-11-20 15:54:57 - 
present)
  - abc Painter expressed intense frustration to zeptester regarding a system issue. (2025-11-20 15:54:57 - 
present)
  - abc Painter is currently using the landscape generator, and it is not working correctly. (2025-11-20 15:54:57 -
present)
  - abc Painter reported that the AI art generation is completely broken. (2025-11-20 15:54:57 - 2025-11-20 
15:54:57)
</FACTS>

In [ ]:
r = zep.graph.search(user_id=user_id, query="Why are there so many goats?", limit=20, scope="edges")
rich.print(r.edges)

[
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:57:15.897Z',
        episodes=['562a9c5f-0b00-47c8-b42b-2a09b792891d'],
        expired_at=None,
        fact='The magic pen tool incorrectly draws goats when attempting to draw boats.',
        invalid_at=None,
        name='ACTUAL_OUTPUT',
        relevance=None,
        score=1.5,
        source_node_uuid='3dea64e4-aa87-4a3a-bb8a-c720f8b869b1',
        target_node_uuid='e0ba2aee-16c0-4bd8-b6cc-5e78eebcf341',
        uuid_='dc4a983f-540b-4cf6-bd80-24098d97d13a',
        valid_at='2024-03-16T14:20:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:57:15.897Z',
        episodes=['562a9c5f-0b00-47c8-b42b-2a09b792891d'],
        expired_at=None,
        fact='The magic pen replaces the drawn boat shape with goats.',
        invalid_at=None,
        name='REPLACES_SHAPE_WITH',
        relevance=None,
        score=1.3333334,
        source_node_uuid='61580874-2014-46b4-9216-987e12860b87',
        target_node_uuid='fc8a15bd-cc8c-498b-92b9-e0fcb78d25d2',
        uuid_='1ec9c813-a447-4d35-b958-112585450b30',
        valid_at='2024-03-16T14:25:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:57:15.897Z',
        episodes=['562a9c5f-0b00-47c8-b42b-2a09b792891d'],
        expired_at=None,
        fact="The support agent addressed the issue titled 'Bug: Magic Pen Tool Drawing Goats Instead of Boats'.",
        invalid_at=None,
        name='HAS_SUBJECT',
        relevance=None,
        score=0.8333333,
        source_node_uuid='80ae1757-97ee-43c1-898b-ba176e3ef2ac',
        target_node_uuid='d9f04ec5-203b-4474-9615-fca0060ba85e',
        uuid_='873dd1c1-84f2-4085-b6d4-1a46d298edab',
        valid_at='2024-03-16T14:22:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:57:15.897Z',
        episodes=['562a9c5f-0b00-47c8-b42b-2a09b792891d'],
        expired_at=None,
        fact='The user, abc Painter, reported the bug where the magic pen tool draws goats instead of boats.',
        invalid_at=None,
        name='REPORTS_ISSUE',
        relevance=None,
        score=0.31666666,
        source_node_uuid='e1076023-b90b-4200-9eca-37b8b353386e',
        target_node_uuid='d9f04ec5-203b-4474-9615-fca0060ba85e',
        uuid_='a2ff1907-7b4f-4a6c-904a-037444556c9b',
        valid_at='2024-03-16T14:20:00Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    )
]

In [9]:
r = zep.graph.search(user_id=user_id, query="what is the reason for the frustation of abc painter", limit=4, scope="edges")
rich.print(r.edges)

[
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:55:02.711Z',
        episodes=['9df7042d-e7f0-4805-9eda-a47fe4b79e82'],
        expired_at='2025-11-20T15:55:28.797Z',
        fact='zeptester greets abc Painter.',
        invalid_at='2025-11-20T15:54:57.689Z',
        name='GREETS',
        relevance=None,
        score=1.25,
        source_node_uuid='a918b27b-28f6-4464-861e-ded2036f2528',
        target_node_uuid='e1076023-b90b-4200-9eca-37b8b353386e',
        uuid_='097c3bd3-6f30-4d78-913b-3269171be0d0',
        valid_at='2025-11-20T15:54:57.689Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:56:05.72Z',
        episodes=['2167f89a-8e19-4502-8bf9-e1849e95a58d'],
        expired_at=None,
        fact='abc Painter is currently using the landscape generator, and it is not working correctly.',
        invalid_at=None,
        name='IS_USING',
        relevance=None,
        score=1.0588236,
        source_node_uuid='e1076023-b90b-4200-9eca-37b8b353386e',
        target_node_uuid='deaa49b6-54d6-4523-8653-1d174b688236',
        uuid_='9be7a135-c0a6-403d-a20e-9a19b13794d7',
        valid_at='2025-11-20T15:54:57.689Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:55:54.267Z',
        episodes=['53f099dd-c52a-4268-89f8-43cf34d30744'],
        expired_at=None,
        fact="The glitch in the system is causing disruption to abc Painter's artistic process.",
        invalid_at=None,
        name='CAUSES_DISRUPTION_TO',
        relevance=None,
        score=0.5769231,
        source_node_uuid='8a2529ef-3d16-48e6-bda8-fc6c69422759',
        target_node_uuid='8f59c8ea-3891-4f71-b997-908861319445',
        uuid_='afb0c9a1-f9dd-4e67-9d49-7ef813f14951',
        valid_at='2025-11-20T15:54:57.689Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    ),
    EntityEdge(
        attributes={},
        created_at='2025-11-20T15:55:02.711Z',
        episodes=['9df7042d-e7f0-4805-9eda-a47fe4b79e82'],
        expired_at='2025-11-20T15:55:28.797Z',
        fact='zeptester welcomes abc Painter to PaintWiz support.',
        invalid_at='2025-11-20T15:54:57.689Z',
        name='WELCOMES',
        relevance=None,
        score=0.5625,
        source_node_uuid='a918b27b-28f6-4464-861e-ded2036f2528',
        target_node_uuid='e1076023-b90b-4200-9eca-37b8b353386e',
        uuid_='112c4c50-3e7e-4c0a-9588-148cf0198638',
        valid_at='2025-11-20T15:54:57.689Z',
        graph_id='e1076023-b90b-4200-9eca-37b8b353386e'
    )
]

In [10]:
memory = zep.thread.get_user_context(thread_id=thread_id)
rich.print(memory.context)

# This is the user summary
<USER_SUMMARY>
The user's name is abc Painter, and their email address is abc@painters.com. Their user ID is abce90d.

The user highly values functional AI art generation, as evidenced by their extreme frustration when the magic pen 
tool failed to generate specified objects (e.g., drawing goats instead of boats on 2024-03-16).

The user has not provided any specific procedural instructions for the AI's assistance.
</USER_SUMMARY>


FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
# NOTE: Facts ending in "present" are currently valid (e.g., "Jane prefers her coffee with milk (2024-01-15 
10:30:00 - present)" means Jane currently prefers coffee with milk)
#       Facts with a past end date used to be valid but are NOT CURRENTLY VALID (e.g., "Jane prefers her coffee 
with milk (2024-01-15 10:30:00 - 2024-06-20 14:00:00)" means Jane no longer prefers coffee with milk)
<FACTS>
  - The glitch in the system is causing disruption to abc Painter's artistic process. (2025-11-20 15:54:57 - 
present)
  - zeptester asked abc Painter to specify which tool or feature is causing the issue. (2025-11-20 15:54:57 - 
present)
  - abc Painter expressed intense frustration to zeptester regarding a system issue. (2025-11-20 15:54:57 - 
present)
  - abc Painter is currently using the landscape generator, and it is not working correctly. (2025-11-20 15:54:57 -
present)
  - abc Painter reported that the AI art generation is completely broken. (2025-11-20 15:54:57 - 2025-11-20 
15:54:57)
</FACTS>